In [3]:
import numpy as np

R, C, M, B, N0, Pmin, Pmax, Rt, dp, T = 4, 4, 3, 20.0, 1e-9, 5.0, 20.0, 75.0, 0.5, 40

np.random.seed(42)
ch = np.random.randint(1, M + 1, (R, C))
p = np.random.uniform(Pmin, Pmax, (R, C))

def G(i, j, u, v):
    return 1e-7 if (i, j) == (u, v) else 1e-8 / (np.hypot(i - u, j - v) ** 3)

def neighbors(i, j):
    return [(u, v) for u in range(max(0, i-1), min(R, i+2))
                   for v in range(max(0, j-1), min(C, j+2)) if (u, v) != (i, j)]

def eval_tp(ch, p):
    tp = np.zeros((R, C))
    for i in range(R):
        for j in range(C):
            I = sum(p[u, v] * G(u, v, i, j) for u, v in neighbors(i, j) if ch[u, v] == ch[i, j])
            tp[i, j] = B * np.log2(1 + (p[i, j] * G(i, j, i, j)) / (N0 + I))
    return tp

print(f"Initial Total Throughput: {eval_tp(ch, p).sum():.2f} Mbps")

for _ in range(T):
    tp = eval_tp(ch, p)
    ch_n, p_n = ch.copy(), p.copy()
    for i in range(R):
        for j in range(C):
            nb = neighbors(i, j)
            ch_n[i, j] = min(range(1, M + 1), key=lambda c: sum(p[u, v] * G(u, v, i, j) for u, v in nb if ch[u, v] == c))
            p_n[i, j] = np.clip(p[i, j] + (dp if tp[i, j] < Rt else -dp), Pmin, Pmax)
    ch, p = ch_n, p_n

final_tp = eval_tp(ch, p)
print(f"Final Total Throughput:   {final_tp.sum():.2f} Mbps")
print("Optimized Channels:\n", ch)
print("Optimized Power (W):\n", np.round(p, 1))

Initial Total Throughput: 1606.31 Mbps
Final Total Throughput:   855.94 Mbps
Optimized Channels:
 [[1 1 3 3]
 [1 3 3 3]
 [3 3 3 3]
 [3 3 3 2]]
Optimized Power (W):
 [[20. 20. 20. 20.]
 [20. 20. 20. 20.]
 [20. 20. 20. 20.]
 [20. 20. 20.  9.]]
